In [326]:
import requests
import torch
from torch.nn import functional as F
import math

from torch.utils.data import TensorDataset, DataLoader

In [327]:
# Loading the dataset

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
print(len(text))

# Set default device to MPS (computations should be made on the GPU)
torch.set_default_device("mps")
# Use faster FP32 matmul kernels where available (noticeable on Apple GPU).
torch.set_float32_matmul_precision("high")

1115394


In [328]:
# Tokenizer

vocab = sorted(set(text))  # Vocabulary
vocab_size = len(vocab)

stoi = {ch: i for i, ch in enumerate(vocab)}  # Char to Int
itos = {i: ch for i, ch in enumerate(vocab)}  # Int to Char

encode = lambda s: [stoi[c] for c in s]  # String -> List[int]
decode = lambda v: "".join([itos[i] for i in v]) # List[Int] -> String

print(encode("hello world"))
print(decode(encode("hello world")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [329]:
# Create (Input, Target) pairs
# Output will be a pair of tensors (X, y) both are a list of tensors of shape (B, L)
def get_batched_data(batch_size, seq_len, text):
    # Encode the text
    encoded_text = torch.tensor(encode(text))

    n = len(encoded_text)

    # Inputs will be a slice of length L and targets will be the next slice of length L to the right
    X = [
        torch.stack([encoded_text[j : j + seq_len] for j in range(i, i + batch_size)])
        for i in range(0, n + 1 - batch_size - seq_len, batch_size)
    ]

    y = [
        torch.stack([
            encoded_text[j + 1 : j + 1 + seq_len] for j in range(i, i + batch_size)]
        )
        for i in range(0, n + 1 - batch_size - seq_len, batch_size)
    ]
  
    return X, y

In [330]:
# Embedding layer
class Embedding:
    def __init__(self, vocab_size, embed_dim):
        self.W = (torch.randn(vocab_size, embed_dim) * 0.1).detach().requires_grad_(True)

    def parameters(self):
        return [self.W]

    def forward(self, X):
        return self.W[X]

In [331]:
# Positional encodings
class PositionalEncodings:
    def __init__(self, seq_len, embed_dim):
        PE = torch.zeros(seq_len, embed_dim)

        pos = torch.arange(seq_len).unsqueeze(1)        # (seq_len, 1)
        div = torch.exp(
            torch.arange(0, embed_dim, 2) * (-math.log(10000) / embed_dim)
        )                                                    # (embed_dim/2,)

        PE[:, 0::2] = torch.sin(pos * div)                  # even dims
        PE[:, 1::2] = torch.cos(pos * div)                  # odd dims

        self.PE = PE.unsqueeze(0)                            # (1, seq_len, embed_dim)

    def forward(self, x):
        return x + self.PE[:, :x.shape[1], :]

In [332]:
# Layer Normalization
class LayerNorm:
    def __init__(self, embed_dim):
        self.gamma = torch.ones(embed_dim, requires_grad=True)
        self.beta = torch.zeros(embed_dim, requires_grad=True)

    def parameters(self):
        return [self.gamma, self.beta]

    def forward(self, X):
        mean = X.mean(dim=-1, keepdim=True)
        std = X.std(dim=-1, keepdim=True, unbiased=False)
        normal = (X - mean) / (std + 1e-5)
        return normal * self.gamma + self.beta

In [333]:
# Multi headed self-attention layer
class MultiHeadSelfAttention:
    def __init__(self, seq_len, embed_dim, num_heads):
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.seq_len = seq_len
        self.head_dim = embed_dim // num_heads
        self.mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool),
            diagonal=1,
        )
        # Maybe add bias?
        self.Wq = (torch.randn(embed_dim, embed_dim) * 0.1).detach().requires_grad_(True)
        self.Wk = (torch.randn(embed_dim, embed_dim) * 0.1).detach().requires_grad_(True)
        self.Wv = (torch.randn(embed_dim, embed_dim) * 0.1).detach().requires_grad_(True)
        self.Wo = (torch.randn(embed_dim, embed_dim) * 0.1).detach().requires_grad_(True)

    def parameters(self):
        return [self.Wq, self.Wk, self.Wv, self.Wo]

    def compute_score(self, Q, K):
        return Q @ torch.transpose(K, 2, 3) / math.sqrt(self.head_dim)

    def forward(self, X):
        Q = X @ self.Wq
        K = X @ self.Wk
        V = X @ self.Wv

        # Splitting into multiple heads
        batch_size, seq_len, _ = X.shape
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = self.compute_score(Q, K)
        scores = scores.masked_fill(self.mask, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        R = attn @ V
        R = R.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)

        return R @ self.Wo

In [334]:
# Feed forward layer
class FeedForward:
    def __init__(self, embed_dim, ff_dim):
        self.W1 = (torch.randn(embed_dim, ff_dim) * 0.1).detach().requires_grad_(True)
        self.b1 = torch.zeros(ff_dim, requires_grad=True)
        self.W2 = (torch.randn(ff_dim, embed_dim) * 0.1).detach().requires_grad_(True)
        self.b2 = torch.zeros(embed_dim, requires_grad=True)

    def parameters(self):
        return [self.W1, self.b1, self.W2, self.b2]

    def forward(self, x):
        return F.relu(x @ self.W1 + self.b1) @ self.W2 + self.b2

In [335]:
# Projection layer
class Projection:
    def __init__(self, embed_dim, vocab_size):
        self.W = (torch.randn(embed_dim, vocab_size) * 0.1).detach().requires_grad_(True)

    def parameters(self):
        return [self.W]

    def forward(self, x):
        return x @ self.W

In [336]:

class Transformer:
    def __init__(self, seq_len, vocab_size, embed_dim, ff_dim, num_heads):
        self.seq_len = seq_len
        self.vocab_size = vocab_size
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // num_heads
        self.E = Embedding(vocab_size, embed_dim)
        self.PE = PositionalEncodings(seq_len, embed_dim)
        self.LN1 = LayerNorm(embed_dim)
        self.MHSA1 = MultiHeadSelfAttention(seq_len, embed_dim, num_heads)
        self.LN2 = LayerNorm(embed_dim)
        self.FF1 = FeedForward(embed_dim, ff_dim)
        self.LN3 = LayerNorm(embed_dim)
        self.MHSA2 = MultiHeadSelfAttention(seq_len, embed_dim, num_heads)
        self.LN4 = LayerNorm(embed_dim)
        self.FF2 = FeedForward(embed_dim, ff_dim)
        self.LN5 = LayerNorm(embed_dim)
        self.MHSA3 = MultiHeadSelfAttention(seq_len, embed_dim, num_heads)
        self.LN6 = LayerNorm(embed_dim)
        self.FF3 = FeedForward(embed_dim, ff_dim)
        self.P = Projection(embed_dim, vocab_size)

    def parameters(self):
        p = []
        p += self.E.parameters()
        p += self.LN1.parameters()
        p += self.MHSA1.parameters()
        p += self.LN2.parameters()
        p += self.FF1.parameters()
        p += self.LN3.parameters()
        p += self.MHSA2.parameters()
        p += self.LN4.parameters()
        p += self.FF2.parameters()
        p += self.LN5.parameters()
        p += self.MHSA3.parameters()
        p += self.LN6.parameters()
        p += self.FF3.parameters()
        p += self.P.parameters()
        return p

    def forward(self, x):
        x = self.E.forward(x)
        x = self.PE.forward(x)
        x = x + self.MHSA1.forward(self.LN1.forward(x))
        x = x + self.FF1.forward(self.LN2.forward(x))
        x = x + self.MHSA2.forward(self.LN3.forward(x))
        x = x + self.FF2.forward(self.LN4.forward(x))
        x = x + self.MHSA3.forward(self.LN5.forward(x))
        x = x + self.FF3.forward(self.LN6.forward(x))
        x = self.P.forward(x)
        return x
        

In [343]:
# Defining the Transformer
# Larger batch => fewer optimizer steps per epoch (watch MPS memory; drop to 512 if needed).
batch_size = 512  # Batch size - B
seq_len = 64 # Length of each input/target - L

T = Transformer(
    seq_len=seq_len, vocab_size=vocab_size, embed_dim=64, ff_dim=256, num_heads=4
)


In [345]:
# Create data

X, y = get_batched_data(
    batch_size, seq_len, text
)  # Fetching inputs X, and targets y

D = int(0.9 * len(X))  # Size of training data

X_train, y_train = torch.stack(X[:D]), torch.stack(y[:D])  # Training data
X_test, y_test = torch.stack(X[D:]), torch.stack(y[D:])  # Test data

print(X_train.shape)


# train_dataset = TensorDataset(X_train, y_train)
# test_dataset = TensorDataset(X_test, y_test)

# train_loader = DataLoader(train_dataset, batch_size=32)

torch.Size([1960, 512, 64])


In [355]:
lr = 0.01

params = T.parameters()

# Training loop
for epoch in range(5):
    it = torch.randperm(X_train.size(0))
    total_loss = torch.zeros(1)
    # Forward pass
    for t in it:
        for param in params:
                param.grad = None
        with torch.autocast(device_type="mps", dtype=torch.float16):
            logits = T.forward(X_train[t])
        # logits = T.forward(X_train[t])
        logits = logits.view(batch_size * seq_len, vocab_size)
        targets = y_train[t].reshape(batch_size * seq_len)
        loss = F.cross_entropy(logits.float(), targets)
        total_loss += loss.detach()
        # Backpropagation
        loss.backward()
        # Gradient descent step
        with torch.no_grad():
            for param in params:
                param -= lr * param.grad
    print("Loss is:", total_loss.item() / X_train.size(0))

Loss is: 1.84609337631537
Loss is: 1.8173911581234057
Loss is: 1.8056619449537628
Loss is: 1.8003470284598215
Loss is: 1.7944379534040178


In [341]:
# Generate
def gen(text, num_tokens):
    print("Original text\n", text)
    # Encode the text
    encoded_text = torch.tensor(encode(text))

    for _ in range(num_tokens):

        X = encoded_text[-seq_len:].unsqueeze(0)  # [L] tensor of last L characters
       
        prob = F.softmax(T.forward(X), dim=-1)[0, -1, :]

        sample = torch.multinomial(prob, num_samples=1)
        
        encoded_text = torch.cat((encoded_text, sample))
        
    print("Generated text is:\n", decode(encoded_text.tolist()))




In [361]:
gen(text[:1000], 250)

Original text
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst